# Model Optimization: Distributed Quantization

In this notebook, we'll apply quantization techniques to our models using distributed processing. Instead of running the quantization on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the quantization on more powerful instances.

This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor
from sagemaker.pytorch.processing import PyTorchProcessor

# Import our utility functions for distributed processing
from sagemaker_processing import run_quantization_job

# Import workshop configuration
from workshop_config import S3_BUCKET, AWS_REGION, SAGEMAKER_ROLE_ARN

## 2. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics from file
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

# Load model information from file
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded information for {len(model_info)} models")

## 3. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 4. Upload Quantization Script to S3

In [ ]:
# Upload the quantization script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'quantization_script.py', 
    S3_BUCKET, 
    'scripts/quantization_script.py'
)

print(f"Uploaded quantization script to s3://{S3_BUCKET}/scripts/quantization_script.py")

## 5. Launch Distributed Quantization Jobs

In [ ]:
# Define the instance type to use for quantization
# For CPU-based quantization, ml.c5.xlarge is a good choice
# For GPU-based quantization, ml.g4dn.xlarge is a good choice
instance_type = "ml.c5.xlarge"

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-quantization",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch quantization jobs for each model
quantization_jobs = {}

for model_key in model_info.keys():
    print(f"\nLaunching quantization job for {model_key}...")
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination="/opt/ml/processing/input/model_info"
        )
    ]
    
    outputs = [
        ProcessingOutput(
            source="/opt/ml/processing/output/quantized_model",
            destination=f's3://{S3_BUCKET}/optimization/outputs/{model_key}/quantized'
        )
    ]
    
    # Run the processing job
    processor.run(
        code="quantization_script.py",
        inputs=inputs,
        outputs=outputs,
        wait=False
    )
    
    # Store the job name
    quantization_jobs[model_key] = processor.latest_job
    print(f"Launched job {processor.latest_job.job_name}")
    
    # Clean up temporary file
    os.remove(f'temp_{model_key}_info.json')

## 6. Monitor Job Status

In [ ]:
# Monitor job status
import time

# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Check job status every 30 seconds
all_completed = False
while not all_completed:
    all_completed = True
    job_statuses = {}
    
    for model_key, job in quantization_jobs.items():
        response = sagemaker_client.describe_processing_job(
            ProcessingJobName=job.job_name
        )
        status = response['ProcessingJobStatus']
        job_statuses[model_key] = status
        
        if status in ['InProgress', 'Stopping']:
            all_completed = False
    
    # Display status table
    status_df = pd.DataFrame({
        'Model': list(job_statuses.keys()),
        'Status': list(job_statuses.values())
    })
    display(status_df)
    
    if not all_completed:
        print("Waiting for jobs to complete...")
        time.sleep(30)
    else:
        print("All jobs completed!")

## 7. Load and Evaluate Quantized Models

In [ ]:
# Function to download a model from S3 to local storage
def download_model_from_s3(s3_uri, local_path):
    """Download model from S3 to local path."""
    # Parse S3 URI
    s3_parts = s3_uri.replace("s3://", "").split("/")
    bucket = s3_parts[0]
    prefix = "/".join(s3_parts[1:])
    
    # Create S3 client
    s3_client = boto3.client('s3')
    
    # List objects in the prefix
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    
    # Download each file
    os.makedirs(local_path, exist_ok=True)
    for obj in response.get('Contents', []):
        key = obj['Key']
        filename = os.path.basename(key)
        if filename:  # Skip directory entries
            local_file = os.path.join(local_path, filename)
            s3_client.download_file(bucket, key, local_file)
    
    print(f"Downloaded model from {s3_uri} to {local_path}")

In [ ]:
# Import utility functions for evaluation
from utils import measure_inference_time, get_model_size, measure_memory_usage
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForMaskedLM

# Function to prepare inputs
def prepare_inputs(model_key, tokenizer, device):
    """Prepare inputs for the model based on task."""
    task = model_info[model_key]["task"]
    sample_input = sample_inputs[model_key]
    
    if task == "sequence-classification" or task == "token-classification":
        inputs = tokenizer(sample_input, return_tensors="pt")
    elif task == "question-answering":
        inputs = tokenizer(sample_input["question"], sample_input["context"], return_tensors="pt")
    elif task == "masked-lm":
        inputs = tokenizer(sample_input, return_tensors="pt")
    else:
        raise ValueError(f"Unsupported task: {task}")
    
    # Move inputs to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    return inputs

In [ ]:
# Evaluate quantized models
quantized_metrics = {}

for model_key in model_info.keys():
    print(f"\nEvaluating quantized model for {model_key}...")
    
    # Define paths
    s3_uri = f's3://{S3_BUCKET}/optimization/outputs/{model_key}/quantized'
    local_path = f'quantized_models/{model_key}'
    
    # Download model
    download_model_from_s3(s3_uri, local_path)
    
    # Load model and tokenizer
    task = model_info[model_key]["task"]
    tokenizer = AutoTokenizer.from_pretrained(local_path)
    
    # Load model based on task
    if task == "sequence-classification":
        model = AutoModelForSequenceClassification.from_pretrained(local_path)
    elif task == "token-classification":
        model = AutoModelForTokenClassification.from_pretrained(local_path)
    elif task == "question-answering":
        model = AutoModelForQuestionAnswering.from_pretrained(local_path)
    elif task == "masked-lm":
        model = AutoModelForMaskedLM.from_pretrained(local_path)
    
    # Move model to device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    
    # Prepare inputs
    inputs = prepare_inputs(model_key, tokenizer, device)
    
    # Measure metrics
    model_size = get_model_size(model)
    inference_time = measure_inference_time(model, inputs)
    memory_usage = measure_memory_usage(model, inputs)
    
    # Run inference to get output
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Store metrics
    quantized_metrics[model_key] = {
        "model_key": model_key,
        "model_name": model_info[model_key]["model_name"],
        "task": model_info[model_key]["task"],
        "model_size": model_size,  # MB
        "inference_time": inference_time,  # ms
        "memory_usage": memory_usage,  # MB
        "quantized_outputs": outputs,
        "quantized_path": local_path
    }
    
    # Print metrics
    print(f"Model size: {model_size:.2f} MB")
    print(f"Inference time: {inference_time:.2f} ms")
    print(f"Memory usage: {memory_usage:.2f} MB")

## 8. Compare Baseline vs. Quantized Models

In [ ]:
# Create comparison data
comparison_data = []

for model_key in model_info.keys():
    if model_key in baseline_metrics and model_key in quantized_metrics:
        model_name = model_info[model_key]["model_name"]
        
        # Add baseline metrics
        comparison_data.append({
            "Model": model_name,
            "Type": "Baseline",
            "Size (MB)": baseline_metrics[model_key]["model_size"],
            "Inference Time (ms)": baseline_metrics[model_key]["inference_time"],
            "Memory Usage (MB)": baseline_metrics[model_key]["memory_usage"]
        })
        
        # Add quantized metrics
        comparison_data.append({
            "Model": model_name,
            "Type": "Quantized",
            "Size (MB)": quantized_metrics[model_key]["model_size"],
            "Inference Time (ms)": quantized_metrics[model_key]["inference_time"],
            "Memory Usage (MB)": quantized_metrics[model_key]["memory_usage"]
        })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)
comparison_df

In [ ]:
# Plot model size comparison
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Size (MB)", hue="Type", data=comparison_df)
plt.title("Model Size Comparison: Baseline vs. Quantized")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot inference time comparison
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Inference Time (ms)", hue="Type", data=comparison_df)
plt.title("Inference Time Comparison: Baseline vs. Quantized")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Plot memory usage comparison
plt.figure(figsize=(12, 6))
sns.barplot(x="Model", y="Memory Usage (MB)", hue="Type", data=comparison_df)
plt.title("Memory Usage Comparison: Baseline vs. Quantized")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 9. Calculate Improvement Percentages

In [ ]:
# Calculate improvement percentages
improvement_data = []

for model_key in model_info.keys():
    if model_key in baseline_metrics and model_key in quantized_metrics:
        model_name = model_info[model_key]["model_name"]
        
        # Calculate size reduction
        baseline_size = baseline_metrics[model_key]["model_size"]
        quantized_size = quantized_metrics[model_key]["model_size"]
        size_reduction = (baseline_size - quantized_size) / baseline_size * 100
        
        # Calculate inference time improvement
        baseline_time = baseline_metrics[model_key]["inference_time"]
        quantized_time = quantized_metrics[model_key]["inference_time"]
        time_improvement = (baseline_time - quantized_time) / baseline_time * 100
        
        # Calculate memory usage reduction
        baseline_memory = baseline_metrics[model_key]["memory_usage"]
        quantized_memory = quantized_metrics[model_key]["memory_usage"]
        memory_reduction = (baseline_memory - quantized_memory) / baseline_memory * 100
        
        improvement_data.append({
            "Model": model_name,
            "Size Reduction (%)": size_reduction,
            "Inference Time Improvement (%)": time_improvement,
            "Memory Usage Reduction (%)": memory_reduction
        })

# Create DataFrame
improvement_df = pd.DataFrame(improvement_data)
improvement_df

## 10. Save Quantized Model Information

In [ ]:
# Save quantized model information (excluding outputs which aren't JSON serializable)
serializable_metrics = {}
for model_key, metrics in quantized_metrics.items():
    serializable_metrics[model_key] = {
        "model_key": metrics["model_key"],
        "model_name": metrics["model_name"],
        "task": metrics["task"],
        "model_size": metrics["model_size"],
        "inference_time": metrics["inference_time"],
        "memory_usage": metrics["memory_usage"],
        "quantized_path": metrics["quantized_path"]
    }

with open('quantized_metrics.json', 'w') as f:
    json.dump(serializable_metrics, f, indent=2)

print("Quantized model metrics saved to quantized_metrics.json")

## 11. Next Steps

Now that we've applied quantization to our models, we'll explore pruning in the next notebook to further reduce model size.